# curbage_fMRI Analyses
## Aging reduces shared event structure at encoding, weakening memory reinstatement in the Posterior Medial Network
### Angelique I. Delarazan, Charan Ranganath, Zachariah M. Reagh


### Set Up

#### Import packages

In [38]:
import numpy as np
import os
import pandas as pd
import pingouin as pg
from scipy.stats import ttest_ind
from scipy.stats import permutation_test
from statsmodels.stats.multitest import multipletests

### Functions

In [39]:
def stat_func(x, y, axis):
    t, _ = ttest_ind(x, y, equal_var=False, axis=axis)
    return t

def stat_func_one_sample(x, axis):
    return np.mean(x, axis=axis)

def load_subject_data(subject, hemi, network, preprocessed_dir, bold_path_template):
    """
    Load neural data for a single subject.
    
    Returns:
        numpy array or None if file not found
    """
    try:
        file_path = os.path.join(preprocessed_dir,
            bold_path_template.format(subject, subject, hemi, network))
        return np.load(file_path)[:min_tr, :]
    except FileNotFoundError:
        return None

def compute_group_isc(subject_data_dict, group_subjects):
    available = [s for s in group_subjects if s in subject_data_dict]
    n = len(available)
    if n < 2:
        return np.nan
    total_sum = np.sum([subject_data_dict[s] for s in available], axis=0)
    fisher_zs = []
    for target_sub in available:
        target_data = subject_data_dict[target_sub]
        avg_comparison = (total_sum - target_data) / (n - 1)
        t_demean = target_data - target_data.mean(axis=0, keepdims=True)
        c_demean = avg_comparison - avg_comparison.mean(axis=0, keepdims=True)
        num = np.sum(t_demean * c_demean, axis=0)
        den = np.where(
            np.sum(t_demean**2, axis=0) * np.sum(c_demean**2, axis=0) > 0,
            np.sqrt(np.sum(t_demean**2, axis=0) * np.sum(c_demean**2, axis=0)),
            np.nan
        )
        vertex_rs = num / den
        fz = np.nanmean(np.arctanh(np.clip(vertex_rs, -0.999, 0.999)))
        if not np.isnan(fz):
            fisher_zs.append(fz)
    return np.mean(fisher_zs) if fisher_zs else np.nan

def circular_shift_test_avg_hemi(subject_data_lh, subject_data_rh, 
                                  group_subjects, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    
    # Observed: compute per hemi, average
    obs_lh = compute_group_isc(subject_data_lh, group_subjects)
    obs_rh = compute_group_isc(subject_data_rh, group_subjects)
    observed_isc = np.mean([obs_lh, obs_rh])
    
    if np.isnan(observed_isc):
        return np.nan, np.nan, np.array([])
    
    first_sub = next(s for s in group_subjects if s in subject_data_lh)
    n_trs = subject_data_lh[first_sub].shape[0]
    
    null_dist = np.zeros(n_iter)
    for i in range(n_iter):
        if (i + 1) % 200 == 0:
            print(f"    Permutation {i + 1}/{n_iter}")
        
        shifted_lh, shifted_rh = {}, {}
        for sub in group_subjects:
            if sub not in subject_data_lh or sub not in subject_data_rh:
                continue
            shift = rng.integers(1, n_trs)
            shifted_lh[sub] = np.roll(subject_data_lh[sub], shift, axis=0)
            shifted_rh[sub] = np.roll(subject_data_rh[sub], shift, axis=0)
        
        null_lh = compute_group_isc(shifted_lh, group_subjects)
        null_rh = compute_group_isc(shifted_rh, group_subjects)
        null_dist[i] = np.mean([null_lh, null_rh])
    
    p_value = (np.sum(null_dist >= observed_isc) + 1) / (n_iter + 1)
    return observed_isc, p_value, null_dist

def group_perm_test(df, value_col='isc', n_iter=5000, seed=42):
    rng = np.random.default_rng(seed)
    older = df[df.group == 'older'][value_col].values
    younger = df[df.group == 'younger'][value_col].values
    obs_t, _ = ttest_ind(older, younger, equal_var=False)

    concat = df[value_col].values
    labels = df['group'].values.copy()
    null_dist = []
    for _ in range(n_iter):
        rng.shuffle(labels)
        t, _ = ttest_ind(
            concat[labels == 'older'],
            concat[labels == 'younger'],
            equal_var=False
        )
        null_dist.append(t)

    null_dist = np.array(null_dist)
    p_perm = (np.sum(np.abs(null_dist) >= np.abs(obs_t)) + 1) / (n_iter + 1)
    return obs_t, p_perm, null_dist

def run_group_perm(
        df, value_col='isc', label='', n_iter=5000, seed=42):
        obs_t, p_perm, null_dist = group_perm_test(df, value_col=value_col, n_iter=n_iter, seed=seed)
    
        older_mean = df[df.group == 'older'][value_col].mean()
        younger_mean = df[df.group == 'younger'][value_col].mean()
        
        result = {
            'comparison': label,
            'older_mean': round(older_mean, 4),
            'younger_mean': round(younger_mean, 4),
            't': round(obs_t, 4),
            'p_perm': round(p_perm, 4),
            'significant': p_perm < 0.05
        }
        return pd.DataFrame([result]), null_dist

def run_corrs(df, neural_col, beh_col, analysis=''):
    results = []
    sub = df
    
    for comp in sub['comparison'].unique():
        for group in sub['group'].unique():
            for method in ['spearman']:
                s = sub[(sub['group'] == group) & (sub['comparison'] == comp)]
                corr = pg.corr(s[neural_col], s[beh_col], method=method)
                results.append({
                    'analysis': analysis,
                    'comparison': comp,
                    'group': group,
                    'method': method,
                    'n': corr['n'].values[0],
                    'r': corr['r'].values[0],
                    'p': corr['p-val'].values[0]
                })
    
    return pd.DataFrame(results).sort_values('p').round(3)

#### Path Directories

In [40]:
base_dir = '/Volumes/omega/curbage_fMRI/'
data_dir = os.path.join(base_dir, 'data/')
roi_dir = os.path.join(data_dir, 'rois/')
beh_dir = os.path.join(data_dir, 'beh/')
preprocessed_dir = os.path.join(data_dir, f'preprocessed/')
analyses_dir = os.path.join(data_dir, f'analyses/')

with open('/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all.txt', 'r') as f:
    subjects = [line.strip() for line in f if line.strip()]

TRshift = 5
TRs = 1.22
min_tr = 1270
desc = 'preproc_bold_motion06_zscore_detrend'
group_dict = {sub: 'older' if int(sub) > 100 else 'younger' for sub in subjects}
bold_path_template = 'sub-{}/func/sub-{}_task-encoding_space-fsaverage_hemi-{}_desc-preproc_bold_motion06_zscore_detrend_network-PMN.npy'
np.random.seed(42)

### Recall Specificity

\begin{equation*}
Specificity Ratio = 1+\frac{{(Detail Units - Gist Units)}} {{(Detail Units + Gist Units)}}
\end{equation*}

In [41]:
beh = pd.read_pickle('~/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-beh_desc-SpecifcityRatio.pkl')
beh['subject'] = beh['subject'].astype(str).str.zfill(2)
beh['specificity_ratio'] = 1+ ((beh['detail_unit']-beh['gist_unit'])/(beh['detail_unit']+beh['gist_unit']))
beh.head()

,subject,group,gist_unit,detail_unit,specificity_ratio
0,03,younger,62,52,0.912281
1,04,younger,73,64,0.934307
2,05,younger,43,38,0.938272
3,06,younger,37,28,0.861538
4,07,younger,46,37,0.891566


In [42]:
pg.ttest(x = beh[(beh['group']=='younger')]['specificity_ratio'],
         y = beh[(beh['group']=='older')]['specificity_ratio'],
         paired=False
)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,2.728309,42.758001,two-sided,0.0092,"[0.03, 0.21]",0.74565,5.275,0.696116


### Pattern Reinstatement

In [43]:
reinstatement = pd.read_pickle(f'~/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-all_space-fsaverage_desc-preproc_bold_motion06_zscore_detrend_5TRshift_PatternReinstatement.pkl')
reinstatement = reinstatement.groupby(['subject', 'network', 'group', 'comparison', 'match'], as_index=False)['fisher_z'].mean()
reinstatement['subject'] = reinstatement['subject'].astype(str).str.zfill(2)

reinstatement = reinstatement.pivot_table(
    index=['subject', 'group', 'network', 'comparison'],
    columns='match',
    values='fisher_z'
).reset_index()

reinstatement['ps'] = reinstatement['matched'] - reinstatement['mismatched']
reinstatement = reinstatement[['subject', 'group', 'network', 'comparison', 'ps']].reset_index(drop=True)
reinstatement.head()

match,subject,group,network,comparison,ps
0,03,younger,PMN,within,-0.010223
1,04,younger,PMN,within,0.002637
2,05,younger,PMN,within,-0.008126
3,06,younger,PMN,within,0.072231
4,07,younger,PMN,within,-0.011994


#### Testing Against 0

In [44]:
ya = reinstatement[(reinstatement['group']=='younger')]['ps'].values
oa = reinstatement[(reinstatement['group']=='older')]['ps'].values

In [45]:
# Testing Against 0
for group, data in [('Younger Adults', ya), ('Older Adults', oa)]:
    print(f"{group}: M = {np.mean(data):.4f}, SD = {np.std(data, ddof=1):.4f}, n = {len(data)}")

res_ya = permutation_test((ya,), stat_func_one_sample, n_resamples=5000, 
                           permutation_type='samples',
                           alternative='greater')

res_oa = permutation_test((oa,), stat_func_one_sample, n_resamples=5000, 
                           permutation_type='samples',
                           alternative='greater')

print(f"Younger Adults: mean = {np.mean(ya):.4f}, p = {res_ya.pvalue:.4f}")
print(f"Older Adults: mean = {np.mean(oa):.4f}, p = {res_oa.pvalue:.4f}")

# For each group
for group, data in [('younger', ya), ('older', oa)]:
    mean_val = np.mean(data)
    cohens_d = mean_val / np.std(data, ddof=1)  # one-sample d = mean / sd
    
    # Bootstrap CI for the mean
    np.random.seed(42)
    boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) 
                  for _ in range(10000)]
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    
    print(f"{group.capitalize()} Adults: mean = {mean_val:.4f}, d = {cohens_d:.2f}, "
          f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

Younger Adults: M = 0.0460, SD = 0.0361, n = 27
Older Adults: M = 0.0240, SD = 0.0290, n = 19
Younger Adults: mean = 0.0460, p = 0.0002
Older Adults: mean = 0.0240, p = 0.0006
Younger Adults: mean = 0.0460, d = 1.27, 95% CI = [0.0324, 0.0589]
Older Adults: mean = 0.0240, d = 0.83, 95% CI = [0.0118, 0.0372]


#### Group Comparison

In [46]:
# Permutation test
res = permutation_test((ya, oa), stat_func, n_resamples=5000, alternative='two-sided')

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(ya)-1)*np.std(ya, ddof=1)**2 + (len(oa)-1)*np.std(oa, ddof=1)**2) / (len(ya)+len(oa)-2))
cohens_d = (np.mean(ya) - np.mean(oa)) / pooled_std

# Bootstrap 95% CI for the mean difference
n_boot = 10000
boot_diffs = []
for _ in range(n_boot):
    boot_ya = np.random.choice(ya, size=len(ya), replace=True)
    boot_oa = np.random.choice(oa, size=len(oa), replace=True)
    boot_diffs.append(np.mean(boot_ya) - np.mean(boot_oa))
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

print(f"Observed t: {ttest_ind(ya, oa, equal_var=False)[0]:.4f}, perm p = {res.pvalue:.4f}")
print(f"Permutation pvalue = {res.pvalue:.4f}")
print(f"Cohen's d = {cohens_d:.2f}")
print(f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

Observed t: 2.2857, perm p = 0.0268
Permutation pvalue = 0.0268
Cohen's d = 0.66
95% CI = [0.0032, 0.0401]


### Intersubject Correlation

In [47]:
isc = pd.read_pickle(f'/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-all_space-fsaverage_desc-{desc}_5TRshift_ISC.pkl')
isc = isc.groupby(['subject', 'network', 'group', 'comparison'], as_index=False)['fisher_z'].mean()
isc = isc.rename(columns={'fisher_z':'isc'})
isc

,subject,network,group,comparison,isc
0,03,PMN,younger,cross,0.134679
1,03,PMN,younger,within,0.185249
2,04,PMN,younger,cross,0.095708
3,04,PMN,younger,within,0.134982
4,05,PMN,younger,cross,0.112329
...,...,...,...,...,...
95,41,PMN,younger,within,0.201321
96,44,PMN,younger,cross,0.042335
97,44,PMN,younger,within,0.057571
98,45,PMN,younger,cross,0.115568


### Circular Shift Permutation 

In [ ]:
n_iter = 1000

older_subs = [s for s in subjects if group_dict[s] == 'older']
younger_subs = [s for s in subjects if group_dict[s] == 'younger']

results = []
null_dists = {}  # store for plotting later

for network in ['PMN']:
    print(f"\nNetwork: {network}")
    
    # Load both hemis separately
    subject_data_lh = {}
    subject_data_rh = {}
    for sub in subjects:
        try:
            lh_path = os.path.join(preprocessed_dir,
                bold_path_template.format(sub, sub, 'lh', network))
            rh_path = os.path.join(preprocessed_dir,
                bold_path_template.format(sub, sub, 'rh', network))
            subject_data_lh[sub] = np.load(lh_path)[:min_tr, :]
            subject_data_rh[sub] = np.load(rh_path)[:min_tr, :]
        except FileNotFoundError:
            print(f"  Missing: sub-{sub}")
            continue

    for group_label, group_subs in [
        ('older', older_subs),
        ('younger', younger_subs),
    ]:
        available = [s for s in group_subs 
                     if s in subject_data_lh and s in subject_data_rh]
        print(f"\n  Testing {group_label} group ({len(available)} subjects)...")

        if len(available) < 3:
            print(f"  Skipping — too few subjects")
            continue

        obs_isc, p_val, null_dist = circular_shift_test_avg_hemi(
            subject_data_lh, 
            subject_data_rh, 
            available, 
            n_iter=n_iter, 
            seed=42
        )
        
        # Store results
        null_dists[(group_label, network)] = null_dist
        
        results.append({
            'network': network,
            'group': group_label,
            'observed_isc_z': obs_isc,
            'observed_isc_r': np.tanh(obs_isc) if not np.isnan(obs_isc) else np.nan,
            'p_value': p_val,
            'significant_05': p_val < 0.05 if not np.isnan(p_val) else False,
            'significant_01': p_val < 0.01 if not np.isnan(p_val) else False,
            'n_subjects': len(available),
            'n_permutations': n_iter,
        })

        sig_marker = "***" if p_val < 0.01 else ("*" if p_val < 0.05 else "n.s.")
        print(f"  ISC = {obs_isc:.4f}, p = {p_val:.4f} {sig_marker}")

sig_results = pd.DataFrame(results)
sig_results


Network: PMN

  Testing older group (23 subjects)...
  ISC = 0.0918, p = 0.0909 n.s.

  Testing younger group (27 subjects)...
  ISC = 0.1456, p = 0.0909 n.s.


,network,group,observed_isc_z,observed_isc_r,p_value,significant_05,significant_01,n_subjects,n_permutations
0,PMN,older,0.091796,0.091539,0.090909,False,False,23,10
1,PMN,younger,0.145563,0.144543,0.090909,False,False,27,10


In [51]:
for comp in ['within', 'cross']:
    sub = isc[isc.comparison == comp]
    for group in ['younger', 'older']:
        vals = sub[sub.group == group]['isc'].values

        # One-sample Cohen's d
        d = np.mean(vals) / np.std(vals, ddof=1)
        
        # Bootstrap CI for the mean
        np.random.seed(42)
        boot_means = [np.mean(np.random.choice(vals, size=len(vals), replace=True)) 
                      for _ in range(10000)]
        ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
        
        print(f"{group.capitalize()} Adult {comp.capitalize()}: "
              f"Cohen's d = {d:.2f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

Younger Adult Within: Cohen's d = 2.93, 95% CI = [0.1267, 0.1635]
Older Adult Within: Cohen's d = 2.47, 95% CI = [0.0771, 0.1067]
Younger Adult Cross: Cohen's d = 2.98, 95% CI = [0.0946, 0.1214]
Older Adult Cross: Cohen's d = 2.28, 95% CI = [0.0831, 0.1184]


#### Group Comparison

In [52]:
# Within-group ISC: OA vs YA
within_result, within_null = run_group_perm(
    isc[isc.comparison == 'within'], value_col='isc', label='within-group_ISC')

# Cross-group ISC: OA vs YA
cross_result, cross_null = run_group_perm(
    isc[isc.comparison == 'cross'], value_col='isc', label='cross-group_ISC')

pd.concat([within_result, cross_result]).reset_index(drop=True)

,comparison,older_mean,younger_mean,t,p_perm,significant
0,within-group_ISC,0.0918,0.1456,-4.3700,0.0002,True
1,cross-group_ISC,0.1008,0.1083,-0.6572,0.5059,False


### Encoding-to-Encoding ISPS

In [53]:
isps_enc_enc = pd.read_pickle(f'/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-all_space-fsaverage_desc-{desc}_5TRshift_EncodingToEncodingISPS.pkl')

isps_enc_enc = isps_enc_enc.pivot_table(
    index=['subject', 'network', 'group', 'comparison'],
    columns='match',
    values='fisher_z'
).reset_index()

isps_enc_enc['isps'] = isps_enc_enc['matched'] - isps_enc_enc['mismatched']
isps_enc_enc

match,subject,network,group,comparison,matched,mismatched,isps
0,03,PMN,younger,cross,0.151839,-0.001222,0.153062
1,03,PMN,younger,within,0.206485,-0.001805,0.208290
2,04,PMN,younger,cross,0.121681,-0.002705,0.124386
3,04,PMN,younger,within,0.170611,-0.003394,0.174005
4,05,PMN,younger,cross,0.096141,-0.001957,0.098098
...,...,...,...,...,...,...,...
95,41,PMN,younger,within,0.179569,-0.004912,0.184482
96,44,PMN,younger,cross,0.055331,0.001245,0.054085
97,44,PMN,younger,within,0.055504,0.002216,0.053288
98,45,PMN,younger,cross,0.132127,-0.000867,0.132994


#### Testing Against 0

In [54]:
ya = isps_enc_enc[(isps_enc_enc['group']=='younger')]['isps'].values
oa = isps_enc_enc[(isps_enc_enc['group']=='older')]['isps'].values

In [55]:
# Testing against 0
for comp in ['within', 'cross']:
    sub = isps_enc_enc[isps_enc_enc.comparison == comp]
    ya = sub[sub.group == 'younger']['isps'].values
    oa = sub[sub.group == 'older']['isps'].values
    
    for group, data in [('Younger', ya), ('Older', oa)]:
        # Permutation test
        res = permutation_test((data,), stat_func_one_sample, n_resamples=5000,
                               permutation_type='samples', alternative='greater')
        
        # Cohen's d
        d = np.mean(data) / np.std(data, ddof=1)
        
        # Bootstrap CI
        boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True))
                      for _ in range(10000)]
        ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
        
        print(f"{group} Adults {comp.capitalize()}: mean = {np.mean(data):.4f}, "
              f"perm p = {res.pvalue:.4f}, d = {d:.2f}, "
              f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

Younger Adults Within: mean = 0.1659, perm p = 0.0002, d = 2.78, 95% CI = [0.1434, 0.1877]
Older Adults Within: mean = 0.0902, perm p = 0.0002, d = 2.91, 95% CI = [0.0782, 0.1030]
Younger Adults Cross: mean = 0.1272, perm p = 0.0002, d = 2.88, 95% CI = [0.1105, 0.1432]
Older Adults Cross: mean = 0.1133, perm p = 0.0002, d = 2.51, 95% CI = [0.0962, 0.1324]


#### Group Comparisons

In [56]:
# Within-group ISC: OA vs YA
within_result, within_null = run_group_perm(
    isps_enc_enc[isps_enc_enc.comparison == 'within'], value_col='isps', label='within-group_ISPS_enc_enc')

# Cross-group ISC: OA vs YA
cross_result, cross_null = run_group_perm(
    isps_enc_enc[isps_enc_enc.comparison == 'cross'], value_col='isps', label='cross-group_ISPS_enc_enc')

pd.concat([within_result, cross_result]).reset_index(drop=True)

,comparison,older_mean,younger_mean,t,p_perm,significant
0,within-group_ISPS_enc_enc,0.0902,0.1659,-5.7365,0.0002,True
1,cross-group_ISPS_enc_enc,0.1133,0.1272,-1.0957,0.2735,False


In [57]:
# After run_group_perm calls
for comp in ['within', 'cross']:
    sub = isps_enc_enc[isps_enc_enc.comparison == comp]
    older = sub[sub.group == 'older']['isps'].values
    younger = sub[sub.group == 'younger']['isps'].values
    
    # Cohen's d
    pooled_std = np.sqrt(((len(younger)-1)*np.std(younger, ddof=1)**2 + 
                          (len(older)-1)*np.std(older, ddof=1)**2) / 
                         (len(younger)+len(older)-2))
    d = (np.mean(younger) - np.mean(older)) / pooled_std
    
    # Bootstrap CI for mean difference
    np.random.seed(42)
    boot_diffs = [np.mean(np.random.choice(younger, size=len(younger), replace=True)) - 
                  np.mean(np.random.choice(older, size=len(older), replace=True)) 
                  for _ in range(10000)]
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    
    print(f"{comp}: Cohen's d = {d:.2f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

within: Cohen's d = 1.55, 95% CI = [0.0496, 0.1001]
cross: Cohen's d = 0.31, 95% CI = [-0.0111, 0.0373]


### Encoding-to-Recall ISPS

In [58]:
isps_enc_rec = pd.read_pickle(f'/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-all_space-fsaverage_desc-{desc}_5TRshift_EncodingToRecallISPS.pkl')

isps_enc_rec = isps_enc_rec.pivot_table(
    index=['subject', 'network', 'group', 'comparison'],
    columns='match',
    values='fisher_z'
).reset_index()

isps_enc_rec['isps'] = isps_enc_rec['matched'] - isps_enc_rec['mismatched']
isps_enc_rec

match,subject,network,group,comparison,matched,mismatched,isps
0,03,PMN,younger,cross,0.027146,0.001293,0.025853
1,03,PMN,younger,within,0.036695,0.001254,0.035441
2,04,PMN,younger,cross,0.028621,0.000057,0.028564
3,04,PMN,younger,within,0.042379,0.000535,0.041844
4,05,PMN,younger,cross,0.005673,0.000365,0.005308
...,...,...,...,...,...,...,...
87,41,PMN,younger,within,0.057960,0.001130,0.056831
88,44,PMN,younger,cross,0.035256,-0.000926,0.036181
89,44,PMN,younger,within,0.026969,-0.000557,0.027526
90,45,PMN,younger,cross,0.066514,-0.001784,0.068297


#### Testing Against 0

In [59]:
# Testing against 0
for comp in ['within', 'cross']:
    sub = isps_enc_rec[isps_enc_rec.comparison == comp]
    ya = sub[sub.group == 'younger']['isps'].values
    oa = sub[sub.group == 'older']['isps'].values
    
    for group, data in [('Younger', ya), ('Older', oa)]:
        # Permutation test
        res = permutation_test((data,), stat_func_one_sample, n_resamples=5000,
                               permutation_type='samples', alternative='greater')
        
        # Cohen's d
        d = np.mean(data) / np.std(data, ddof=1)
        
        # Bootstrap CI
        boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True))
                      for _ in range(10000)]
        ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
        
        print(f"{group} Adults {comp.capitalize()}: mean = {np.mean(data):.4f}, "
              f"perm p = {res.pvalue:.4f}, d = {d:.2f}, "
              f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

Younger Adults Within: mean = 0.0455, perm p = 0.0002, d = 1.61, 95% CI = [0.0353, 0.0561]
Older Adults Within: mean = 0.0365, perm p = 0.0002, d = 2.03, 95% CI = [0.0285, 0.0441]
Younger Adults Cross: mean = 0.0403, perm p = 0.0002, d = 1.97, 95% CI = [0.0325, 0.0476]
Older Adults Cross: mean = 0.0385, perm p = 0.0002, d = 1.81, 95% CI = [0.0292, 0.0477]


#### Group Comparisons

In [60]:
# Within-group ISC: OA vs YA
within_result, within_null = run_group_perm(
    isps_enc_rec[isps_enc_rec.comparison == 'within'], value_col='isps', label='within-group_ISPS_enc_rec')

# Cross-group ISC: OA vs YA
cross_result, cross_null = run_group_perm(
    isps_enc_rec[isps_enc_rec.comparison == 'cross'], value_col='isps', label='cross-group_ISPS_enc_rec')

pd.concat([within_result, cross_result]).reset_index(drop=True)

,comparison,older_mean,younger_mean,t,p_perm,significant
0,within-group_ISPS_enc_rec,0.0365,0.0455,-1.3217,0.1956,False
1,cross-group_ISPS_enc_rec,0.0385,0.0403,-0.2880,0.7788,False


In [61]:
# After run_group_perm calls
for comp in ['within', 'cross']:
    sub = isps_enc_rec[isps_enc_rec.comparison == comp]
    older = sub[sub.group == 'older']['isps'].values
    younger = sub[sub.group == 'younger']['isps'].values
    
    # Cohen's d
    pooled_std = np.sqrt(((len(younger)-1)*np.std(younger, ddof=1)**2 + 
                          (len(older)-1)*np.std(older, ddof=1)**2) / 
                         (len(younger)+len(older)-2))
    d = (np.mean(younger) - np.mean(older)) / pooled_std
    
    # Bootstrap CI for mean difference
    np.random.seed(42)
    boot_diffs = [np.mean(np.random.choice(younger, size=len(younger), replace=True)) - 
                  np.mean(np.random.choice(older, size=len(older), replace=True)) 
                  for _ in range(10000)]
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    
    print(f"{comp}: Cohen's d = {d:.2f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

within: Cohen's d = 0.37, 95% CI = [-0.0041, 0.0220]
cross: Cohen's d = 0.09, 95% CI = [-0.0103, 0.0136]


### Recall-to-Recall ISPS

In [62]:
isps_rec_rec = pd.read_pickle(f'/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-all_space-fsaverage_desc-{desc}_5TRshift_RecallToRecallISPS.pkl')

isps_rec_rec = isps_rec_rec.pivot_table(
    index=['subject', 'network', 'group', 'comparison'],
    columns='match',
    values='fisher_z'
).reset_index()

isps_rec_rec['isps'] = isps_rec_rec['matched'] - isps_rec_rec['mismatched']
isps_rec_rec

match,subject,network,group,comparison,matched,mismatched,isps
0,03,PMN,younger,cross,0.037601,-0.000457,0.038058
1,03,PMN,younger,within,0.066395,-0.001343,0.067738
2,04,PMN,younger,cross,0.030371,0.000324,0.030047
3,04,PMN,younger,within,0.053915,-0.000129,0.054044
4,05,PMN,younger,cross,-0.030497,0.001304,-0.031801
...,...,...,...,...,...,...,...
87,41,PMN,younger,within,0.059194,-0.001711,0.060905
88,44,PMN,younger,cross,0.029334,-0.001899,0.031233
89,44,PMN,younger,within,0.053868,-0.001688,0.055556
90,45,PMN,younger,cross,0.061559,-0.000994,0.062554


#### Testing Against 0

In [63]:
# Testing against 0
for comp in ['within', 'cross']:
    sub = isps_rec_rec[isps_rec_rec.comparison == comp]
    ya = sub[sub.group == 'younger']['isps'].values
    oa = sub[sub.group == 'older']['isps'].values
    
    for group, data in [('Younger', ya), ('Older', oa)]:
        # Permutation test
        res = permutation_test((data,), stat_func_one_sample, n_resamples=5000,
                               permutation_type='samples', alternative='greater')
        
        # Cohen's d
        d = np.mean(data) / np.std(data, ddof=1)
        
        # Bootstrap CI
        boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True))
                      for _ in range(10000)]
        ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
        
        print(f"{group} Adults {comp.capitalize()}: mean = {np.mean(data):.4f}, "
              f"perm p = {res.pvalue:.4f}, d = {d:.2f}, "
              f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

Younger Adults Within: mean = 0.0478, perm p = 0.0002, d = 1.97, 95% CI = [0.0387, 0.0568]
Older Adults Within: mean = 0.0375, perm p = 0.0002, d = 1.81, 95% CI = [0.0288, 0.0469]
Younger Adults Cross: mean = 0.0300, perm p = 0.0002, d = 1.47, 95% CI = [0.0223, 0.0372]
Older Adults Cross: mean = 0.0374, perm p = 0.0002, d = 1.63, 95% CI = [0.0271, 0.0474]


#### Group Comparisons

In [64]:
# Within-group ISC: OA vs YA
within_result, within_null = run_group_perm(
    isps_rec_rec[isps_rec_rec.comparison == 'within'], value_col='isps', label='within-group_ISPS_rec_rec')

# Cross-group ISC: OA vs YA
cross_result, cross_null = run_group_perm(
    isps_rec_rec[isps_rec_rec.comparison == 'cross'], value_col='isps', label='cross-group_ISPS_rec_rec')

pd.concat([within_result, cross_result]).reset_index(drop=True)

,comparison,older_mean,younger_mean,t,p_perm,significant
0,within-group_ISPS_rec_rec,0.0375,0.0478,-1.5530,0.1268,False
1,cross-group_ISPS_rec_rec,0.0374,0.0300,1.1237,0.2657,False


In [65]:
# After run_group_perm calls
for comp in ['within', 'cross']:
    sub = isps_rec_rec[isps_rec_rec.comparison == comp]
    older = sub[sub.group == 'older']['isps'].values
    younger = sub[sub.group == 'younger']['isps'].values
    
    # Cohen's d
    pooled_std = np.sqrt(((len(younger)-1)*np.std(younger, ddof=1)**2 + 
                          (len(older)-1)*np.std(older, ddof=1)**2) / 
                         (len(younger)+len(older)-2))
    d = (np.mean(younger) - np.mean(older)) / pooled_std
    
    # Bootstrap CI for mean difference
    np.random.seed(42)
    boot_diffs = [np.mean(np.random.choice(younger, size=len(younger), replace=True)) - 
                  np.mean(np.random.choice(older, size=len(older), replace=True)) 
                  for _ in range(10000)]
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    
    print(f"{comp}: Cohen's d = {d:.2f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

within: Cohen's d = 0.45, 95% CI = [-0.0028, 0.0230]
cross: Cohen's d = -0.34, 95% CI = [-0.0199, 0.0054]


### Neuropsychological Tests

In [66]:
neuropsychs = pd.read_pickle('/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-beh_desc-Neuropsych.pkl')
neuropsychs['subject'] = neuropsychs['subject'].astype(str).str.zfill(2)
neuropsychs.head()

,subject,moca_total,craft21_verbatim,craft21_paraphrase,craft21Delayed_verbatim,craft21Delayed_paraphrase,mint_total
0,106,29,30,21,24,17,32
1,107,29,36,24,36,24,32
2,108,29,23,15,20,17,32
3,110,25,25,17,24,17,32
4,111,27,24,18,22,17,32


In [67]:
neuro_tests = ['moca_total', 'craft21_verbatim', 'craft21_paraphrase', 
         'craft21Delayed_verbatim', 'craft21Delayed_paraphrase', 'mint_total']

neuropsychs[neuro_tests].describe().round(2)

,moca_total,craft21_verbatim,craft21_paraphrase,craft21Delayed_verbatim,craft21Delayed_paraphrase,mint_total
count,22.00,22.00,22.00,22.00,22.00,22.00
mean,27.55,24.73,18.45,21.77,17.41,31.27
std,2.13,5.84,3.39,7.00,3.79,1.32
min,23.00,13.00,10.00,7.00,8.00,28.00
25%,27.00,23.00,17.00,19.00,15.25,31.00
50%,28.50,25.00,18.00,21.00,17.00,32.00
75%,29.00,28.00,21.00,25.75,20.75,32.00
max,30.00,36.00,24.00,36.00,24.00,32.00


### Correlations: Neural Measures to Recall Specificity

In [68]:
neural_dfs = {
    'reinstatement': reinstatement,
    'isc': isc,
    'isps_enc_enc': isps_enc_enc,
    'isps_enc_rec': isps_enc_rec,
    'isps_rec_rec': isps_rec_rec,
}

neural_specificity = {name: df.merge(beh, on=['subject', 'group'], how='inner') 
          for name, df in neural_dfs.items()}
neural_specificity

{'reinstatement':    subject    group network comparison        ps  gist_unit  detail_unit  \
 0       03  younger     PMN     within -0.010223         62           52   
 1       04  younger     PMN     within  0.002637         73           64   
 2       05  younger     PMN     within -0.008126         43           38   
 3       06  younger     PMN     within  0.072231         37           28   
 4       07  younger     PMN     within -0.011994         46           37   
 5       10  younger     PMN     within  0.047085        107           97   
 6      102    older     PMN     within  0.017227         92           65   
 7      103    older     PMN     within -0.011610         38           17   
 8      106    older     PMN     within  0.070775         40           22   
 9      108    older     PMN     within  0.018100         43           18   
 10      11  younger     PMN     within  0.082309        110           98   
 11     110    older     PMN     within  0.074103         6

In [69]:
reinstatement_beh_corr = run_corrs(neural_specificity['reinstatement'], 'ps', 'specificity_ratio', analysis='ps')
isc_beh_corr = run_corrs(neural_specificity['isc'], 'isc', 'specificity_ratio', analysis='isc')
enc_enc_beh_corr = run_corrs(neural_specificity['isps_enc_enc'], 'isps', 'specificity_ratio', analysis='isps_enc_enc')
enc_rec_beh_corr = run_corrs(neural_specificity['isps_enc_rec'], 'isps', 'specificity_ratio', analysis='isps_enc_rec')
rec_rec_beh_corr = run_corrs(neural_specificity['isps_rec_rec'], 'isps', 'specificity_ratio', analysis='isps_rec_rec')

neural_specificity_corr = pd.concat([reinstatement_beh_corr, isc_beh_corr, enc_enc_beh_corr, enc_rec_beh_corr, rec_rec_beh_corr], ignore_index=True)
neural_specificity_corr

,analysis,comparison,group,method,n,r,p
0,ps,within,younger,spearman,27,-0.060,0.765
1,ps,within,older,spearman,19,0.012,0.960
2,isc,within,younger,spearman,27,0.380,0.051
3,isc,cross,younger,spearman,27,0.306,0.120
4,isc,cross,older,spearman,20,-0.111,0.640
5,isc,within,older,spearman,20,-0.035,0.885
6,isps_enc_enc,cross,younger,spearman,27,0.225,0.260
7,isps_enc_enc,within,younger,spearman,27,0.211,0.292
8,isps_enc_enc,cross,older,spearman,20,-0.191,0.420
9,isps_enc_enc,within,older,spearman,20,-0.018,0.940


In [70]:
df = neural_specificity_corr.copy()
# Apply FDR correction separately for each age group
corrected_results = []

for group in ['younger', 'older']:
    df_group = df[df['group'] == group].copy()
    
    # Get p-values for this group
    p_values = df_group['p'].values
    
    rejected, p_corrected, _, _ = multipletests(
        p_values, 
        alpha=0.05, 
        method='fdr_bh'
    )
    
    df_group['p_FDR'] = p_corrected
    df_group['significant_FDR'] = rejected
    
    corrected_results.append(df_group)

df_corrected = pd.concat(corrected_results, ignore_index=True)

correlations = df_corrected.sort_values(['analysis', 'group']).reset_index(drop=True)
correlations

,analysis,comparison,group,method,n,r,p,p_FDR,significant_FDR
0,isc,cross,older,spearman,20,-0.111,0.640,0.96000,False
1,isc,within,older,spearman,20,-0.035,0.885,0.96000,False
2,isc,within,younger,spearman,27,0.380,0.051,0.45900,False
3,isc,cross,younger,spearman,27,0.306,0.120,0.52560,False
4,isps_enc_enc,cross,older,spearman,20,-0.191,0.420,0.96000,False
5,isps_enc_enc,within,older,spearman,20,-0.018,0.940,0.96000,False
6,isps_enc_enc,cross,younger,spearman,27,0.225,0.260,0.52560,False
7,isps_enc_enc,within,younger,spearman,27,0.211,0.292,0.52560,False
8,isps_enc_rec,cross,older,spearman,19,-0.128,0.601,0.96000,False
9,isps_enc_rec,within,older,spearman,19,-0.104,0.673,0.96000,False


### Correlations: ISCs and ISPSs to Pattern Reinstatement

In [71]:
reinstatement_corr = reinstatement[['subject', 'group', 'network', 'ps']]

neural_dfs = {
    'isc': isc,
    'isps_enc_enc': isps_enc_enc,
    'isps_enc_rec': isps_enc_rec,
    'isps_rec_rec': isps_rec_rec,
}

neural_ps = {name: df.merge(reinstatement_corr, on=['subject', 'group', 'network'], how='inner') 
          for name, df in neural_dfs.items()}
neural_ps

{'isc':    subject network    group comparison       isc        ps
 0       03     PMN  younger      cross  0.134679 -0.010223
 1       03     PMN  younger     within  0.185249 -0.010223
 2       04     PMN  younger      cross  0.095708  0.002637
 3       04     PMN  younger     within  0.134982  0.002637
 4       05     PMN  younger      cross  0.112329 -0.008126
 ..     ...     ...      ...        ...       ...       ...
 87      41     PMN  younger     within  0.201321  0.091301
 88      44     PMN  younger      cross  0.042335 -0.002418
 89      44     PMN  younger     within  0.057571 -0.002418
 90      45     PMN  younger      cross  0.115568  0.094025
 91      45     PMN  younger     within  0.128787  0.094025
 
 [92 rows x 6 columns],
 'isps_enc_enc': match subject network    group comparison   matched  mismatched      isps  \
 0          03     PMN  younger      cross  0.151839   -0.001222  0.153062   
 1          03     PMN  younger     within  0.206485   -0.001805  0.208290 

In [72]:
isc_ps_corr = run_corrs(neural_ps['isc'], 'isc', 'ps', analysis='isc')
enc_enc_ps_corr = run_corrs(neural_ps['isps_enc_enc'], 'isps', 'ps', analysis='isps_enc_enc')
enc_rec_ps_corr = run_corrs(neural_ps['isps_enc_rec'], 'isps', 'ps', analysis='isps_enc_rec')
rec_rec_ps_corr = run_corrs(neural_ps['isps_rec_rec'], 'isps', 'ps', analysis='isps_rec_rec')

neural_ps_corr = pd.concat([isc_ps_corr, enc_enc_ps_corr, enc_rec_ps_corr, rec_rec_ps_corr], ignore_index=True)
neural_ps_corr

,analysis,comparison,group,method,n,r,p
0,isc,cross,younger,spearman,27,0.436,0.023
1,isc,within,younger,spearman,27,0.432,0.024
2,isc,cross,older,spearman,19,0.407,0.084
3,isc,within,older,spearman,19,0.407,0.084
4,isps_enc_enc,within,younger,spearman,27,0.593,0.001
5,isps_enc_enc,cross,older,spearman,19,0.672,0.002
6,isps_enc_enc,cross,younger,spearman,27,0.556,0.003
7,isps_enc_enc,within,older,spearman,19,0.565,0.012
8,isps_enc_rec,within,younger,spearman,27,0.609,0.001
9,isps_enc_rec,cross,older,spearman,19,0.498,0.030


In [73]:
df = neural_ps_corr.copy()
# Apply FDR correction separately for each age group
corrected_results = []

for group in ['younger', 'older']:
    df_group = df[df['group'] == group].copy()
    
    # Get p-values for this group
    p_values = df_group['p'].values
    
    rejected, p_corrected, _, _ = multipletests(
        p_values, 
        alpha=0.05, 
        method='fdr_bh'
    )
    
    df_group['p_FDR'] = p_corrected
    df_group['significant_FDR'] = rejected
    
    corrected_results.append(df_group)

df_corrected = pd.concat(corrected_results, ignore_index=True)

correlations = df_corrected.sort_values(['analysis', 'group']).reset_index(drop=True)
correlations

,analysis,comparison,group,method,n,r,p,p_FDR,significant_FDR
0,isc,cross,older,spearman,19,0.407,0.084,0.112000,False
1,isc,within,older,spearman,19,0.407,0.084,0.112000,False
2,isc,cross,younger,spearman,27,0.436,0.023,0.038400,True
3,isc,within,younger,spearman,27,0.432,0.024,0.038400,True
4,isps_enc_enc,cross,older,spearman,19,0.672,0.002,0.016000,True
5,isps_enc_enc,within,older,spearman,19,0.565,0.012,0.048000,True
6,isps_enc_enc,within,younger,spearman,27,0.593,0.001,0.004000,True
7,isps_enc_enc,cross,younger,spearman,27,0.556,0.003,0.008000,True
8,isps_enc_rec,cross,older,spearman,19,0.498,0.030,0.080000,False
9,isps_enc_rec,within,older,spearman,19,0.177,0.468,0.534857,False


### Demographics

In [74]:
demographics = pd.read_pickle('/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/curbage_fMRI/github/data/curbage_fMRI_sub-all_group-all_task-beh_desc-Demograhpics.pkl')
demographics['subject'] = demographics['subject'].astype(str).str.zfill(2)
demographics.head()

,subject,group,age,gender,ethnicity
0,03,younger,26,Female,Prefer not to answer
1,04,younger,21,Female,Asian
2,05,younger,23,Male,Caucasian / White
3,06,younger,27,Male,Asian
4,07,younger,21,Female,Asian


In [75]:
for group in ['younger', 'older']:
    demographics_subset = demographics[demographics['group'] == group]
    print(f"\n--- {group.capitalize()} Adults (n={len(demographics_subset)}) ---")
    print(f"Age: M={demographics_subset['age'].mean():.2f}, SD={demographics_subset['age'].std():.2f}, range={demographics_subset['age'].min()}-{demographics_subset['age'].max()}")
    print(f"\nGender:\n{demographics_subset['gender'].value_counts().to_string()}")
    print(f"\nEthnicity:\n{demographics_subset['ethnicity'].value_counts().to_string()}")


--- Younger Adults (n=27) ---
Age: M=21.96, SD=2.83, range=18-28

Gender:
gender
Female                  18
Male                     8
Prefer not to answer     1

Ethnicity:
ethnicity
Asian                       12
Caucasian / White            5
Prefer not to answer         4
Not listed here              3
African American / Black     3

--- Older Adults (n=23) ---
Age: M=68.43, SD=6.82, range=60-83

Gender:
gender
Male      12
Female    11

Ethnicity:
ethnicity
Caucasian / White       18
Asian                    2
Not listed here          2
Prefer not to answer     1
